Setting up Connection and checking all data in azure

In [ ]:
!curl ifconfig.me

In [ ]:
# Install unixODBC
!apt-get update
!apt-get install -y unixodbc unixodbc-dev

# Install pyodbc
!pip install pyodbc

# Install ODBC Driver 17 for SQL Server
!curl https://packages.microsoft.com/keys/microsoft.asc | apt-key add -
!curl https://packages.microsoft.com/config/ubuntu/$(lsb_release -rs)/prod.list | tee /etc/apt/sources.list.d/mssql-release.list
!apt-get update
!ACCEPT_EULA=Y apt-get install -y msodbcsql17


In [ ]:


 

import urllib
from sqlalchemy import create_engine
from sqlalchemy.sql import text

username = 'group9studentuser'  # Replace with your Azure SQL admin username
password = 'PasswordGroup9'        # Replace with your Azure SQL password
server = 'team9azureuser.database.windows.net'  # Replace with your Azure SQL server name
database = 'StudemtDB_9'  # Replace with your Azure SQL database

# Define connection parameters
driver = "ODBC Driver 17 for SQL Server"  # Ensure this driver is installed
 
# Build the ODBC connection string
odbc_str = f"DRIVER={{{driver}}};SERVER={server};PORT=1433;DATABASE={database};UID={username};PWD={password}"
 
# Encode the connection string for SQLAlchemy
connection_url = f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(odbc_str)}"
 
# Create SQLAlchemy engine
engine = create_engine(connection_url)
 

with engine.connect() as connection:
    result = connection.execute(text("SELECT GETDATE();"))
    for row in result:
        print("Connected! Server date and time:", row[0])

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_type = 'BASE TABLE';
    """))
    for row in result:
        print(row)


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name, column_name, data_type
        FROM information_schema.columns
        ORDER BY table_name;
    """))
    for row in result:
        print(row)


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 'business_df' AS table_name, COUNT(*) FROM business_df
        UNION ALL
        SELECT 'user_df', COUNT(*) FROM user_df
        UNION ALL
        SELECT 'review_df', COUNT(*) FROM review_df
        UNION ALL
        SELECT 'tip_df', COUNT(*) FROM tip_df
        UNION ALL
        SELECT 'checkin_df', COUNT(*) FROM checkin_df
        UNION ALL
        SELECT 'business_categories', COUNT(*) FROM business_categories;
    """))
    for row in result:
        print(row)


In [ ]:

import pandas as pd
from sqlalchemy import text

# Load business_df
with engine.connect() as conn:
    business_df = pd.read_sql(text("SELECT TOP 10 * FROM business_df;"), conn)

# Load review_df
with engine.connect() as conn:
    review_df = pd.read_sql(text("SELECT TOP 10 * FROM review_df;"), conn)

# Load user_df
with engine.connect() as conn:
    user_df = pd.read_sql(text("SELECT TOP 10 * FROM user_df;"), conn)

# Load tip_df
with engine.connect() as conn:
    tip_df = pd.read_sql(text("SELECT TOP 10 * FROM tip_df;"), conn)

# Load checkin_df
with engine.connect() as conn:
    checkin_df = pd.read_sql(text("SELECT TOP 10 * FROM checkin_df;"), conn)

# Load business_categories
with engine.connect() as conn:
    business_categories = pd.read_sql(text("SELECT TOP 10 * FROM business_categories;"), conn)


In [ ]:
print(" business_df"); display(business_df)
print(" review_df"); display(review_df)
print(" user_df"); display(user_df)
print(" tip_df"); display(tip_df)
print(" checkin_df"); display(checkin_df)
print(" business_categories"); display(business_categories)


In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            COUNT(*) AS total_rows, 
            COUNT(category) AS non_null_categories
        FROM business_categories;
    """))
    for row in result:
        print(row)



## Step 0: Data Validation and Type Preparation

Before performing any analysis, we explored all tables and validated column types to ensure they are suitable for querying. Some key columns were found to be stored as strings or integers and needed to be cast to proper types like `INT`, `FLOAT`, or `TIME`.

This step helps avoid SQL errors and improves the quality of insights extracted from the Yelp data.


### Step 0.1: Cast `attr_RestaurantsPriceRange2` to INT


In [ ]:
with engine.connect() as conn:
    df_price = pd.read_sql(text("""
        SELECT business_id, name, 
               TRY_CAST(attr_RestaurantsPriceRange2 AS INT) AS price_level
        FROM business_df
        WHERE attr_RestaurantsPriceRange2 IS NOT NULL;
    """), conn)

df_price.head()


### Step 0.2: Cast `stars` to FLOAT from `review_df`


In [ ]:
with engine.connect() as conn:
    df_stars = pd.read_sql(text("""
        SELECT review_id, business_id, user_id,
               CAST(stars AS FLOAT) AS star_rating
        FROM review_df;
    """), conn)

df_stars.head()


### Step 0.3: Convert `hours_Friday_open` and `hours_Friday_close` to TIME


In [ ]:
with engine.connect() as conn:
    df_hours = pd.read_sql(text("""
        SELECT business_id, name,
               TRY_CAST(hours_Friday_open AS TIME) AS open_time,
               TRY_CAST(hours_Friday_close AS TIME) AS close_time
        FROM business_df
        WHERE hours_Friday_open IS NOT NULL AND hours_Friday_close IS NOT NULL;
    """), conn)

df_hours.head()


### Step 0.4: Filter Only Open Businesses


In [ ]:
with engine.connect() as conn:
    df_open = pd.read_sql(text("""
        SELECT business_id, name, stars
        FROM business_df
        WHERE is_open = 1;
    """), conn)

df_open.head()


### Step 0.5: Keep Long Reviews for Text Analysis


In [ ]:
with engine.connect() as conn:
    df_reviews_text = pd.read_sql(text("""
        SELECT review_id, business_id, stars, text
        FROM review_df
        WHERE LEN(text) > 100
        ORDER BY LEN(text) DESC
        OFFSET 0 ROWS FETCH NEXT 10 ROWS ONLY;
    """), conn)

df_reviews_text.head()


## Step 1 – Market Distribution: Top Cities by Restaurant Count

Before narrowing down to a specific city, we examined which cities have the highest number of restaurants listed on Yelp.
This helps identify competitive and high-activity markets. Nashville stands out as one of the top restaurant hubs in this dataset.


In [ ]:
with engine.connect() as conn:
    df_city_counts = pd.read_sql(text("""
        SELECT 
            city,
            COUNT(*) AS restaurant_count
        FROM business_df
        WHERE is_open = 1
        GROUP BY city
        ORDER BY restaurant_count DESC;
    """), conn)

df_city_counts.head(20)


**Insight:**  
Cities like Philadelphia, Tampa, and Nashville have a high concentration of restaurants. With over 2,500 open restaurants, Nashville is a strong market to explore further.

**Action:**  
Given its high restaurant volume and vibrant food scene, Nashville is selected as the target city for deeper analysis in this project.


### Step 1 – Query 2: Top-Rated Cuisines in Nashville

To determine the most successful restaurant types in Nashville, we analyze the average ratings of each cuisine category.
Only categories with more than 10 restaurants are considered to ensure statistical relevance. This helps identify which cuisine would be ideal for a new restaurant venture in the city.


In [ ]:
with engine.connect() as conn:
    df_q1_nashville = pd.read_sql(text("""
        SELECT 
            c.category,
            AVG(CAST(b.stars AS FLOAT)) AS avg_rating,
            COUNT(*) AS num_places
        FROM business_df b
        JOIN business_categories c ON b.business_id = c.business_id
        WHERE LOWER(b.city) = 'nashville'
              AND b.is_open = 1
              AND c.category IS NOT NULL
        GROUP BY c.category
        HAVING COUNT(*) > 10
        ORDER BY avg_rating DESC;
    """), conn)

df_q1_nashville.head(10)


In [ ]:
with engine.connect() as conn:
    df_q2_zip_nashville = pd.read_sql(text("""
        SELECT 
            postal_code,
            COUNT(*) AS num_restaurants,
            SUM(review_count) AS total_reviews,
            AVG(CAST(stars AS FLOAT)) AS avg_rating
        FROM business_df
        WHERE LOWER(city) = 'nashville'
              AND is_open = 1
              AND postal_code IS NOT NULL
        GROUP BY postal_code
        ORDER BY total_reviews DESC;
    """), conn)

df_q2_zip_nashville.head(10)


**Insight:**  
The highest-rated categories in Nashville are niche, experience-driven services like photography, Pilates, and martial arts, rather than traditional restaurants. Food-related categories with high ratings include barre classes and bus tours, suggesting that combining dining with unique experiences could resonate well.The most restaurant-dense ZIP code (37203) has nearly 100,000 reviews, indicating high foot traffic and competition, while 37206 has the highest average rating (4.16), suggesting a market that values quality over quantity.

**Action:**  
Consider a hybrid dining concept that merges food with an experiential element, such as cooking classes or themed events, to differentiate from traditional restaurants and tap into Nashville’s demand for engaging activities.If aiming for visibility, target 37203; if prioritizing a premium reputation, focus on 37206. Avoid oversaturated areas like 37214, which has lower ratings.


### Step 1 – Query 3: Most Popular Times for Customer Check-Ins in Nashville

To identify peak customer hours, we analyze check-in times in Nashville using the `checkin_df` table. Understanding when customers are most active helps optimize operating hours and service planning for a new restaurant.


In [ ]:
with engine.connect() as conn:
    df_q3_checkin_hour = pd.read_sql(text("""
        SELECT 
            DATEPART(HOUR, checkin_time) AS hour_of_day,
            COUNT(*) AS total_checkins
        FROM checkin_df c
        JOIN business_df b ON c.business_id = b.business_id
        WHERE LOWER(b.city) = 'nashville'
        GROUP BY DATEPART(HOUR, checkin_time)
        ORDER BY total_checkins DESC;
    """), conn)

df_q3_checkin_hour.head(10)


**Insight:**  
The busiest check-in hours are midnight, 11 PM, and 5–7 PM, showing strong late-night demand alongside traditional dinner peaks.

**Action:**  
Extend operating hours to at least midnight and introduce a happy hour (5–7 PM) to attract both after-work and nightlife crowds.


### Step 1 – Query 4: Longest Reviews in Nashville

Long reviews often reflect highly engaged or passionate customers. 
Analyzing these can reveal:
- Strong sentiment (positive or negative)
- What aspects customers care most about
- Which restaurants get detailed feedback

This query retrieves the longest reviews written for Nashville businesses.

In [ ]:
with engine.connect() as conn:
    df_q4_long_reviews = pd.read_sql(text("""
        SELECT TOP 10 
            r.review_id,
            b.name AS business_name,
            CAST(r.stars AS FLOAT) AS rating,
            r.text,
            LEN(r.text) AS review_length
        FROM review_df r
        JOIN business_df b ON r.business_id = b.business_id
        WHERE LOWER(b.city) = 'nashville'
        ORDER BY LEN(r.text) DESC;
    """), conn)

df_q4_long_reviews



**Insight:**  
Long Nashville reviews show extreme emotions - either raving about great food/service or ranting about bad experiences. Customers focus on specific dishes, staff interactions, and atmosphere.

**Action:**  
Train staff using real review examples. Design "review-worthy" moments like signature dishes or chef meet-and-greets. Quickly respond to all detailed reviews.

### Step 1 – Query 5: Average Rating by Price Range in Nashville

We analyze how restaurant pricing (from Yelp’s `attr_RestaurantsPriceRange2`) correlates with average ratings in Nashville. This guides how premium or affordable the restaurant should be.


In [ ]:
with engine.connect() as conn:
    df_q5_price_rating = pd.read_sql(text("""
        SELECT 
            TRY_CAST(attr_RestaurantsPriceRange2 AS INT) AS price_level,
            COUNT(*) AS num_restaurants,
            AVG(CAST(stars AS FLOAT)) AS avg_rating
        FROM business_df
        WHERE LOWER(city) = 'nashville'
              AND attr_RestaurantsPriceRange2 IS NOT NULL
        GROUP BY TRY_CAST(attr_RestaurantsPriceRange2 AS INT)
        ORDER BY price_level;
    """), conn)

df_q5_price_rating


**Insight:**  
Mid-priced restaurants (price levels 2–3) have the best balance of popularity and ratings (avg. 3.7), while premium-priced spots (level 4) see fewer reviews and lower satisfaction.

**Action:**  
Position the restaurant in the mid-range ($11–30) to align with customer expectations and avoid pricing pitfalls of high-end dining.

### Step 1 – Query 6: Most Reviewed Restaurants in Nashville

This identifies the restaurants that generate the most public attention. It highlights establishments with a strong customer base and viral reach.


In [ ]:
with engine.connect() as conn:
    df_q6_most_reviewed = pd.read_sql(text("""
        SELECT 
            name, 
            review_count, 
            CAST(stars AS FLOAT) AS avg_rating
        FROM business_df
        WHERE LOWER(city) = 'nashville'
        ORDER BY review_count DESC
        OFFSET 0 ROWS FETCH NEXT 10 ROWS ONLY;
    """), conn)

df_q6_most_reviewed


**Insight:**  
The most-reviewed restaurants (e.g., Hattie B’s Hot Chicken, Biscuit Love) are local staples with strong branding, Southern comfort food focus, and ratings above 4.0.

**Action:**  
Study these successful brands for menu inspiration and branding strategies, but differentiate with a modern twist or fusion concept.

### Step 1 – Query 7: Highest Rated Small Restaurants in Nashville

This finds “hidden gems” — restaurants with very high ratings but relatively few reviews. These may represent exceptional service or quality that hasn’t gone mainstream yet.


In [ ]:
with engine.connect() as conn:
    df_q7_hidden_gems = pd.read_sql(text("""
        SELECT 
            name,
            review_count,
            CAST(stars AS FLOAT) AS avg_rating
        FROM business_df
        WHERE LOWER(city) = 'nashville'
              AND is_open = 1
              AND review_count BETWEEN 10 AND 50
              AND stars >= 4.5
        ORDER BY avg_rating DESC, review_count ASC;
    """), conn)

df_q7_hidden_gems


**Insight:**  
Small businesses with 10–50 reviews and 4.5+ ratings thrive in specialized niches, but few are food-related, indicating an opportunity for high-quality, under-the-radar dining concepts.

**Action:**  
Explore underserved niches (e.g., health-focused, globally inspired comfort food) and emphasize authenticity to attract discerning customers.

### Step 1 – Query 8: Most Active Yelp Users in Nashville

This query identifies the most active Yelp users in Nashville based on the number of reviews they've written. These users can influence local dining trends and should be watched or even targeted for early promotions.


In [ ]:
with engine.connect() as conn:
    df_q8_top_users = pd.read_sql(text("""
        SELECT 
            u.user_id,
            u.name,
            u.review_count,
            u.average_stars
        FROM user_df u
        JOIN review_df r ON u.user_id = r.user_id
        JOIN business_df b ON r.business_id = b.business_id
        WHERE LOWER(b.city) = 'nashville'
        GROUP BY u.user_id, u.name, u.review_count, u.average_stars
        ORDER BY u.review_count DESC
        OFFSET 0 ROWS FETCH NEXT 10 ROWS ONLY;
    """), conn)

df_q8_top_users


**Insight:**  
The most active Yelp users in Nashville write thousands of reviews, with average ratings around 3.5, suggesting they are influential but not overly critical.

**Action:**  
Engage these users early with exclusive previews or tastings to generate buzz and gain constructive feedback.

### Step 1 – Query 9: Restaurants Offering Delivery and Takeout in Nashville

This query checks how many businesses in Nashville offer delivery and takeout. It helps assess whether including these services is necessary or optional.


In [ ]:
with engine.connect() as conn:
    df_q9_delivery_takeout = pd.read_sql(text("""
        SELECT 
            SUM(CASE WHEN attr_RestaurantsDelivery = 1 THEN 1 ELSE 0 END) AS offers_delivery,
            SUM(CASE WHEN attr_RestaurantsTakeOut = 1 THEN 1 ELSE 0 END) AS offers_takeout,
            COUNT(*) AS total_restaurants
        FROM business_df
        WHERE LOWER(city) = 'nashville' AND is_open = 1;
    """), conn)

df_q9_delivery_takeout


**Insight:**  
Only 30% of Nashville restaurants offer delivery, while 50% provide takeout, indicating that takeout is a lower-risk, higher-adoption model.

**Action:**  
Launch with a takeout-first model to test demand before investing in delivery infrastructure.

### Step 1 – Query 10: Alcohol Service Among Nashville Restaurants

This query shows how common alcohol service is among Nashville restaurants, helping decide whether to include it in the business model.


In [ ]:
with engine.connect() as conn:
    df_q10_alcohol = pd.read_sql(text("""
        SELECT 
            attr_Alcohol,
            COUNT(*) AS num_restaurants,
            AVG(CAST(stars AS FLOAT)) AS avg_rating
        FROM business_df
        WHERE LOWER(city) = 'nashville' 
              AND is_open = 1
              AND attr_Alcohol IS NOT NULL
        GROUP BY attr_Alcohol
        ORDER BY num_restaurants DESC;
    """), conn)

df_q10_alcohol


**Insight:**  
Restaurants with full bars are most common, but those offering only beer/wine have the highest ratings (3.88), suggesting that a curated beverage program may enhance reputation without the complexity of a full bar.

**Action:**  
Feature a craft beer and wine selection to elevate the dining experience while keeping operations manageable.

**Conclusion**  
Based on the data analysis, Nashville presents a strong opportunity for a mid-range Southern comfort food restaurant with a modern twist, ideally located in ZIP 37206 for quality-focused diners or 37203 for higher foot traffic. The concept should emphasize craft beer/wine, takeout-friendly options, and late-night service (until midnight) to capitalize on peak demand periods, while incorporating unique experiential elements to generate positive reviews and differentiate from competitors. Prioritizing staff training based on common review complaints and creating Instagrammable dishes will help build strong word-of-mouth in this review-driven market.